In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import contextlib
import copy
import io
import os
import pickle
import queue
import threading
from concurrent.futures import ThreadPoolExecutor
from datetime import date

import cma
import numpy as np
import pandas as pd
from sklearn.metrics import root_mean_squared_error

from aquacrop import AquaCrop, Crop, Soil, Weather
from aquacrop_slovenia import config
from aquacrop_slovenia.diagnostics import (
    kge,
    kge_alpha,
    kge_beta,
    kge_r,
    mkge,
    mkge_alpha,
    nse,
)
from aquacrop_slovenia.parameters_running_rakican import (
    rakican_curve_number,
    rakican_groundwater,
    rakican_initial_cond,
    rakican_maize_params,
    rakican_management,
    rakican_readily_evaporable_water,
    rakican_soil_layers,
)
from aquacrop_slovenia.plots import plot_yield_timeseries_comparison
from aquacrop_slovenia.reading_data import (
    get_co2_for_aquacrop,
    get_station_weather,
    get_yield,
    get_yield_for_comparison,
)

# Rakican — leave-one-year-out cross-validation

Same two-stage CMA-ES calibration as `Rakican - Calibration.ipynb` (biomass mKGE, then
grain mKGE on top of the biomass-calibrated parameters), but repeated once per
observed year: each fold holds one year out of both the biomass and grain training
data, calibrates on the remaining years, and evaluates the resulting parameters on the
held-out year only. Pooling every fold's held-out prediction gives an out-of-sample
(cross-validated) estimate of model performance, comparable against the in-sample fit
from `Rakican - Calibration.ipynb`.

This repeats the full CMA-ES search (same `popsize`/`maxiter` as the base notebook)
once per fold, so it takes roughly (number of years) times as long as the base
notebook. Progress is checkpointed to disk after every fold so an interrupted run can
be resumed by re-running the notebook.

In [ ]:
LOCATION = "rakican"
MAX_WORKERS = os.cpu_count()

simulation_periods = [
    {
        "start_date": date(year, 1, 1),
        "end_date": date(year, 12, 31),
        "planting_date": date(year, 4, 20),
        "is_seeding_year": True,
    }
    for year in range(1993, 2024)
    if year not in [1998, 2023, 2017]
]

temperatures, eto, rain = get_station_weather(355)
historical_co2 = get_co2_for_aquacrop("historical")

weather = Weather(
    location=LOCATION,
    temperatures=temperatures,
    eto_values=eto,
    rainfall_values=rain,
    record_type=1,
    first_day=1,
    first_month=1,
    first_year=1993,
    co2_records=historical_co2,
)

soil = Soil(
    name=f"{LOCATION} soil",
    description=f"{LOCATION} silt loam soil",
    soil_layers=rakican_soil_layers,
    curve_number=rakican_curve_number,
    readily_evaporable_water=rakican_readily_evaporable_water,
)

observed_biomass = get_yield_for_comparison(LOCATION, "biomass", "A", "N3")
observed_grain = get_yield_for_comparison(LOCATION, "grain", "A", "N3")

INTEGER_PARAMS = {name for name, val in rakican_maize_params.items() if isinstance(val, int)}

In [ ]:
worker_dirs = []
for i in range(MAX_WORKERS):
    d = config.MODEL_RUNNING_DIR / f"loo_rakican_{i}"
    d.mkdir(parents=True, exist_ok=True)
    worker_dirs.append(d)

dir_queue = queue.Queue()
for d in worker_dirs:
    dir_queue.put(d)

print(f"Worker directories ready: {MAX_WORKERS}")

## Shared calibration machinery

In [ ]:
# Stage 1: biomass-shaping parameters (same search space as the base calibration notebook)
STAGE1_PARAM_NAMES = [
    "gdd_emergence",
    "gdd_max_rooting",
    "gdd_senescence",
    "gdd_maturity",
    "gdd_flowering",
    "gdd_flowering_length",
    "max_canopy_cover",
    "max_rooting_depth",
]
STAGE1_BOUNDS = [
    (5, 100),        # gdd_emergence
    (800, 1200),     # gdd_max_rooting
    (800, 1200),     # gdd_senescence
    (800, 1200),     # gdd_maturity
    (350, 550),      # gdd_flowering
    (80, 170),       # gdd_flowering_length
    (0.85, 0.97),    # max_canopy_cover
    (0.5, 2.5),      # max_rooting_depth
]
STAGE1_LOWER = np.array([b[0] for b in STAGE1_BOUNDS])
STAGE1_UPPER = np.array([b[1] for b in STAGE1_BOUNDS])
STAGE1_X0 = (STAGE1_LOWER + STAGE1_UPPER) / 2.0

# Stage 2: yield-shaping parameters, applied on top of the stage-1 (biomass) result
STAGE2_PARAM_NAMES = ["harvest_index", "gdd_hi_start"]
STAGE2_BOUNDS = [
    (0.46, 0.65),    # harvest_index
    (300, 500),      # gdd_hi_start
]
STAGE2_LOWER = np.array([b[0] for b in STAGE2_BOUNDS])
STAGE2_UPPER = np.array([b[1] for b in STAGE2_BOUNDS])
STAGE2_X0 = (STAGE2_LOWER + STAGE2_UPPER) / 2.0

print(f"Stage 1 params: {STAGE1_PARAM_NAMES}")
print(f"Stage 2 params: {STAGE2_PARAM_NAMES}")

In [ ]:
def run_simulation(crop_params, working_dir):
    """Run one AquaCrop simulation over the full study period and return the seasonal results."""
    crop = Crop(name=f"{LOCATION} maize", description="cma-es loo", params=crop_params)
    simulation = AquaCrop(
        simulation_periods=simulation_periods,
        crop=crop,
        soil=soil,
        management=rakican_management,
        initial_conditions=rakican_initial_cond,
        climate=weather,
        ground_water=rakican_groundwater,
        working_dir=working_dir,
        need_daily_output=False,
        need_seasonal_output=True,
        need_harvest_output=False,
        need_evaluation_output=False,
    )
    with contextlib.redirect_stdout(io.StringIO()):
        results = simulation.run()
    return results["season"][["Year1", "BioMass", "Y(dry)"]].rename(
        columns={"Year1": "year", "BioMass": "biomass_modeled", "Y(dry)": "grain_modeled"}
    )


def build_params(param_names, param_values, base_params):
    """Apply param_names/param_values on top of base_params, rounding integer params."""
    crop_params = copy.deepcopy(base_params)
    for name, val in zip(param_names, param_values):
        crop_params[name] = int(round(val)) if name in INTEGER_PARAMS else float(val)
    return crop_params


def compute_metrics(observed_df, seasonal, modeled_col, prefix):
    # dropna: some years are missing an observed value — skip those rather than
    # feeding NaN into the metrics.
    merged = observed_df.merge(seasonal[["year", modeled_col]], on="year").dropna(
        subset=["yield", modeled_col]
    )
    y_obs = merged["yield"].values
    y_mod = merged[modeled_col].values
    return {
        f"{prefix}_rmse":       root_mean_squared_error(y_obs, y_mod),
        f"{prefix}_nse":        nse(y_mod, y_obs),
        f"{prefix}_kge":        kge(y_mod, y_obs),
        f"{prefix}_mkge":       mkge(y_mod, y_obs),
        f"{prefix}_kge_r":      kge_r(y_mod, y_obs),
        f"{prefix}_kge_beta":   kge_beta(y_mod, y_obs),
        f"{prefix}_kge_alpha":  kge_alpha(y_mod, y_obs),
        f"{prefix}_mkge_alpha": mkge_alpha(y_mod, y_obs),
    }


def evaluate_params(param_names, param_values, base_params, obs_biomass_df, obs_grain_df, working_dir):
    """Run one candidate on top of base_params and score it against the given observed subsets.

    `obs_biomass_df`/`obs_grain_df` are passed in explicitly (rather than closed over)
    so a fold can score candidates against its training years only, or its held-out
    year only, using the same function.
    """
    crop_params = build_params(param_names, param_values, base_params)
    nan_metrics = {
        f"{prefix}_{s}": np.nan
        for prefix in ("biomass", "grain")
        for s in ["rmse", "nse", "kge", "mkge", "kge_r", "kge_beta", "kge_alpha", "mkge_alpha"]
    }
    try:
        seasonal = run_simulation(crop_params, working_dir)
    except Exception:
        return nan_metrics
    return {
        **compute_metrics(obs_biomass_df, seasonal, "biomass_modeled", "biomass"),
        **compute_metrics(obs_grain_df,   seasonal, "grain_modeled",   "grain"),
    }

In [ ]:
SIGMA0 = 0.02


def parallel_evaluate(objective_fn, solutions):
    """Evaluate a list of candidate solutions in parallel using ThreadPoolExecutor."""
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        return list(executor.map(objective_fn, solutions))


def run_cmaes(objective_fn, x0, lower, upper, seed, maxiter=20):
    """Run CMA-ES for one objective function and return the CMAEvolutionStrategy object."""
    options = {
        "bounds": [lower.tolist(), upper.tolist()],
        "CMA_stds": (upper - lower).tolist(),
        "popsize": MAX_WORKERS,
        "maxiter": maxiter,
        "tolx": 1e-5,
        "tolfun": 1e-5,
        "seed": seed,
        "verbose": -9,  # quiet: this runs once per fold, per-iteration logs would be excessive
    }
    es = cma.CMAEvolutionStrategy(x0, SIGMA0, options)
    while not es.stop():
        solutions = es.ask()
        fitnesses = parallel_evaluate(objective_fn, solutions)
        penalty = max((f for f in fitnesses if np.isfinite(f)), default=1e6) * 10
        fitnesses = [penalty if not np.isfinite(f) else f for f in fitnesses]
        es.tell(solutions, fitnesses)
    return es

## Leave-one-year-out loop

In [ ]:
def make_fold_objective(param_names, base_params, obs_biomass_df, obs_grain_df, scalar_fn):
    """Return an objective function that evaluates one candidate against a fold's training data."""
    def objective(param_values):
        working_dir = dir_queue.get()
        try:
            r = evaluate_params(param_names, param_values, base_params, obs_biomass_df, obs_grain_df, working_dir)
        finally:
            dir_queue.put(working_dir)
        return scalar_fn(r)
    return objective

In [ ]:
LOO_YEARS = sorted(set(observed_biomass["year"]) & set(observed_grain["year"]))
print(f"Leave-one-year-out folds: {len(LOO_YEARS)} ({LOO_YEARS[0]}-{LOO_YEARS[-1]})")

CHECKPOINT_PATH = config.RESULTS_DIR / "loo_rakican_checkpoint.pkl"
if CHECKPOINT_PATH.exists():
    with open(CHECKPOINT_PATH, "rb") as f:
        fold_results = pickle.load(f)
    print(f"Resuming: {len(fold_results)} fold(s) already computed")
else:
    fold_results = []

done_years = {r["held_out_year"] for r in fold_results}

In [ ]:
for i, held_out_year in enumerate(LOO_YEARS):
    if held_out_year in done_years:
        continue

    train_biomass = observed_biomass[observed_biomass["year"] != held_out_year].reset_index(drop=True)
    train_grain = observed_grain[observed_grain["year"] != held_out_year].reset_index(drop=True)
    test_biomass = observed_biomass[observed_biomass["year"] == held_out_year].reset_index(drop=True)
    test_grain = observed_grain[observed_grain["year"] == held_out_year].reset_index(drop=True)

    print(f"\n=== Fold {i + 1}/{len(LOO_YEARS)}: held-out year {held_out_year} ===")

    # Stage 1: biomass mKGE, trained on every year except held_out_year
    stage1_objective = make_fold_objective(
        STAGE1_PARAM_NAMES, rakican_maize_params, train_biomass, train_grain,
        lambda r: -r["biomass_mkge"],  # negate: CMA-ES minimises
    )
    es1 = run_cmaes(stage1_objective, STAGE1_X0, STAGE1_LOWER, STAGE1_UPPER, seed=42 + i)
    biomass_calibrated_params = build_params(STAGE1_PARAM_NAMES, es1.result.xbest, rakican_maize_params)
    print(f"  stage 1 (biomass mKGE, train): {-es1.result.fbest:.4f}")

    # Stage 2: grain mKGE, trained on every year except held_out_year, on top of stage 1
    stage2_objective = make_fold_objective(
        STAGE2_PARAM_NAMES, biomass_calibrated_params, train_biomass, train_grain,
        lambda r: -r["grain_mkge"],  # negate: CMA-ES minimises
    )
    es2 = run_cmaes(stage2_objective, STAGE2_X0, STAGE2_LOWER, STAGE2_UPPER, seed=10 + i)
    final_params = build_params(STAGE2_PARAM_NAMES, es2.result.xbest, biomass_calibrated_params)
    print(f"  stage 2 (grain mKGE, train):   {-es2.result.fbest:.4f}")

    # Evaluate the fold's calibrated parameters on the held-out year only
    working_dir = dir_queue.get()
    try:
        seasonal = run_simulation(final_params, working_dir)
    finally:
        dir_queue.put(working_dir)

    held_out_row = seasonal[seasonal["year"] == held_out_year]
    biomass_pred = held_out_row["biomass_modeled"].iloc[0] if not held_out_row.empty else np.nan
    grain_pred = held_out_row["grain_modeled"].iloc[0] if not held_out_row.empty else np.nan

    fold_results.append({
        "held_out_year": held_out_year,
        "biomass_obs": test_biomass["yield"].iloc[0] if not test_biomass.empty else np.nan,
        "biomass_pred": biomass_pred,
        "grain_obs": test_grain["yield"].iloc[0] if not test_grain.empty else np.nan,
        "grain_pred": grain_pred,
        "stage1_best": dict(zip(STAGE1_PARAM_NAMES, es1.result.xbest)),
        "stage2_best": dict(zip(STAGE2_PARAM_NAMES, es2.result.xbest)),
        "stage1_fbest": -es1.result.fbest,
        "stage2_fbest": -es2.result.fbest,
    })
    done_years.add(held_out_year)

    with open(CHECKPOINT_PATH, "wb") as f:
        pickle.dump(fold_results, f)

print(f"\nCompleted {len(fold_results)}/{len(LOO_YEARS)} folds")

## Pooled out-of-sample evaluation

In [ ]:
loo_df = pd.DataFrame(fold_results).sort_values("held_out_year").reset_index(drop=True)
loo_df[["held_out_year", "biomass_obs", "biomass_pred", "grain_obs", "grain_pred", "stage1_fbest", "stage2_fbest"]]

In [ ]:
def pooled_metrics(obs_col, pred_col, prefix):
    """Metrics computed on the pooled held-out predictions — one out-of-sample prediction
    per year, each from the fold that did not train on that year."""
    d = loo_df.dropna(subset=[obs_col, pred_col])
    y_obs = d[obs_col].values
    y_mod = d[pred_col].values
    return {
        f"{prefix}_rmse":       root_mean_squared_error(y_obs, y_mod),
        f"{prefix}_nse":        nse(y_mod, y_obs),
        f"{prefix}_kge":        kge(y_mod, y_obs),
        f"{prefix}_mkge":       mkge(y_mod, y_obs),
        f"{prefix}_kge_r":      kge_r(y_mod, y_obs),
        f"{prefix}_kge_beta":   kge_beta(y_mod, y_obs),
        f"{prefix}_kge_alpha":  kge_alpha(y_mod, y_obs),
        f"{prefix}_mkge_alpha": mkge_alpha(y_mod, y_obs),
    }


loo_metrics = {
    **pooled_metrics("biomass_obs", "biomass_pred", "biomass"),
    **pooled_metrics("grain_obs",   "grain_pred",   "grain"),
}
print("Pooled leave-one-year-out (out-of-sample) metrics:")
pd.Series(loo_metrics).round(3)

Compare `loo_metrics` above (out-of-sample) against the in-sample metrics saved by
`Rakican - Calibration.ipynb` (`cmaes_rakican_calibration_params_2.pkl`) — a
substantially worse LOO score indicates the full-data calibration is overfitting.

In [ ]:
loo_model_grain = loo_df.rename(columns={"held_out_year": "Year1", "grain_pred": "Y(dry)"})[["Year1", "Y(dry)"]]
plot_yield_timeseries_comparison(loo_model_grain, get_yield(LOCATION, "grain", "A", "N3"), "Y(dry)", LOCATION)

In [ ]:
loo_model_biomass = loo_df.rename(columns={"held_out_year": "Year1", "biomass_pred": "BioMass"})[["Year1", "BioMass"]]
plot_yield_timeseries_comparison(loo_model_biomass, get_yield(LOCATION, "biomass", "A", "N3"), "BioMass", LOCATION)

## Per-fold calibrated parameters

In [ ]:
stage1_params_df = pd.DataFrame(
    [r["stage1_best"] for r in fold_results], index=[r["held_out_year"] for r in fold_results]
)
stage1_params_df.index.name = "held_out_year"
print("Stage 1 (biomass) calibrated parameters per fold — spread indicates sensitivity to which year is left out:")
stage1_params_df.sort_index().round(3)

In [ ]:
stage2_params_df = pd.DataFrame(
    [r["stage2_best"] for r in fold_results], index=[r["held_out_year"] for r in fold_results]
)
stage2_params_df.index.name = "held_out_year"
print("Stage 2 (grain) calibrated parameters per fold:")
stage2_params_df.sort_index().round(3)

## Save results

In [ ]:
output_path = config.RESULTS_DIR / "loo_rakican_calibration.pkl"
loo_df.to_pickle(output_path)
print(f"Saved {len(loo_df)} fold(s) to {output_path}")

metrics_path = config.RESULTS_DIR / "loo_rakican_calibration_metrics.pkl"
pd.Series(loo_metrics).to_pickle(metrics_path)
print(f"Saved pooled LOO metrics to {metrics_path}")